In [0]:
%sql
CREATE OR REPLACE TEMP VIEW core_deposits AS
select
  eomdate,

  CASE WHEN bdsp_flag = 'Y' THEN 'SNV_Brokered_BDSP' 
    ELSE y15_class END AS y15_class,
  sum(ledger_balance) as ledger_balance,
  sum(ledger_balance)  as spot_balance,
  sum(op_balance) as operational_balance,
  sum(non_op_balance)as nonoperational_balance,
  sum(ics_ledger_balance) as ics_reciprocal_balance,
  gl_bs_acct,
  '--' as  pnfp_gl_bs_acct,
  counterparty_type,
  maturity_bucket,
  legacy_bank,
  CASE
    WHEN collateral_flag = 'N' THEN counterparty_type
    ELSE 'Secured'
  end as gl_view
from
  finance.reporting.depositsplus_eom
WHERE
  eom_flag = 'Y'
    AND asofdate > '2026-07-01'
    AND (internal_elim_flag = 'N' OR bdsp_flag = 'Y')
        AND (product_type <> 'Brokered' OR src_ac_tp_code IN ( '194',' 196', '199'))
        GROUP BY ALL ;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW brokered_deposits AS 

WITH src AS (
SELECT
  w.Last_Business_Day_Of_Month AS asofdate,
  w.counterparty_name AS account_name,
  TRY_CAST(w.account_number AS STRING) AS account_number,
  CONCAT(w.contract_id, '-', w.tranche) AS accountid,
  w.tranche,
  'Brokered' AS deposit_type,
  'Brokered' AS product_type,
  CASE
    WHEN sub_instrument_type = 'Brokered NMD' THEN '439'
    WHEN sub_instrument_type = 'Brokered CD' THEN '092'
    ELSE NULL
  END AS src_ac_tp_code,
  sub_instrument_type AS src_ac_tp_desc,
  'N' AS account_analysis_flag,
  w.current_balance AS ledger_balance,
  NULL AS ics_reciprocal_balance,
  w.all_in_rate AS interest_rate,
  NULL AS earnings_credit_rate,
  CAST(NULL AS STRING) AS client_id,
  'I' AS client_type,
  'Individual ' AS counterparty_type,
  NULL AS tax_id,
  CAST(NULL AS STRING) AS naics_code,
  'N' AS subsidiary_flag,
  'N' AS collateral_flag,
  'Include' AS gl_flag,
  NULL AS ics_recip_gl_bs_acct,
  w.gl_bs_acct AS pnfp_gl_bs_acct,
  gl_bs_acct AS gl_bs_acct,
  w.End_of_Month AS eomdate,
  w.Last_Business_Day_Of_Month AS eobmdate,
  w.maturity_date AS account_maturity_date,
  w.issuance_date AS account_open_date,
  w.maturity_date AS account_close_date,
  DATEDIFF( w.maturity_date , w.Last_Business_Day_Of_Month ) as days_to_maturity,
  w.entity AS legacy_bank

FROM
  finance.reporting.wholesalefunding_vw w

WHERE
  w.as_of_date = w.Last_Business_Day_Of_Month
  AND w.primary_instrument_type = 'Brokered Deposit'
  AND w.as_of_date > '2026-05-30')


SELECT 
  asofdate as EOMDATE,

  CASE 
    WHEN src_ac_tp_code = '092'
        THEN  CONCAT(legacy_bank,'_', 'Brokered_TIME')
        ELSE CONCAT(legacy_bank,'_', 'Brokered_NMD') 
            END AS y15_class,
  SUM(ledger_balance) as  ledger_balance,
  null as spot_balance,
  null as operational_balance,
  null as nonoperational_balance,
  null as ics_reciprocal_balance,
  gl_bs_acct,
  null as pnfp_gl_bs_acct,
counterparty_type,


  CASE
    WHEN
       days_to_maturity <= 30 THEN 'LT 30 Days'
    WHEN
      days_to_maturity > 30
      AND days_to_maturity <= 90
    THEN
      '31-90 Days'
    WHEN
      days_to_maturity > 90
      AND days_to_maturity <= 180 
    THEN
      '91-180 Days'
    WHEN
      days_to_maturity > 180
      AND days_to_maturity <= 365
    THEN
      '181-365 Days'
        ELSE 
          'GT 365 Days'
  END AS maturity_bucket,
legacy_bank,
'Individual' as gl_view


FROM src GROUP BY ALL ;;


SELECT * FROM brokered_deposits



Databricks visualization. Run in Databricks to view.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW deposits_by_y15_class AS 

SELECT * FROM core_deposits
UNION ALL 
SELECT * FROM brokered_deposits ;

    SELECT * FROM deposits_by_y15_class

Databricks visualization. Run in Databricks to view.


--Take y15 index table and recreate the 5 tab excel work book (1 data tab)
    -- The Actual Schedule
    -- The Deliverables tab
    -- The Detailed Tab
    -- Deposits Recon Tab
    --Deposits by Y/15 Class 

    

